In [1]:
from main import run_bot
from demo import run_demo
import config

In [2]:

from config import MIN_R2, OUTPUTS_DIR, TOP_N_HIGHLIGHT, create_run_dirs
from broker.connection import connect_ib
from broker.data import fetch_prices, fetch_prices_free
from broker.orders import calculate_position_size, execute_order, get_portfolio_value
from analysis.universe import fetch_company_metadata, get_sp500_tickers
from analysis.fundamentals import fetch_fundamentals, score_fundamentals, save_fundamentals_csv
from analysis.correlations import compute_correlations, get_top_correlated_pairs, get_top_inverse_pairs
from analysis.model import predict_price
from analysis.signals import generate_signals
from reporting.charts import (plot_correlation_matrix, plot_market_cap_bars,
                               plot_market_cap_series, plot_prediction_analysis,
                               plot_price_series)
from reporting.report import print_report, save_signals_csv


In [3]:
n_tickers = None    # int = top N by market cap | None = full S&P 500 | 'FALLBACK_TICKERS' = hardcoded top-20
mode = 'paper'      # demo | paper | live | signals
execute_trades=True
save_plots = True

In [4]:

print("\nCreate local folders")
run_dir, gen_dir, corr_dir = create_run_dirs()

print("\nFetching S&P 500 tickers and market_caps")
tickers, market_caps = get_sp500_tickers(n=n_tickers)

print("\nFetch stock prices")
prices_df = fetch_prices_free(tickers)
if prices_df.empty or len(prices_df.columns) < 5:
    print("✗ Insufficient data. Aborting.")
    return

print("\nCalculating correlations")
corr_matrix, returns = compute_correlations(prices_df)
top_pairs     = get_top_correlated_pairs(corr_matrix, top_n=10)
inverse_pairs = get_top_inverse_pairs(corr_matrix, top_n=10)

print("\nGenerate the signals table")
signals_df = generate_signals(prices_df, returns, corr_matrix)

print("\nEnrich signals with company name, sector, founded year, market cap (B)")
company_meta = fetch_company_metadata(list(prices_df.columns), market_caps)
signals_df = signals_df.merge(
    company_meta.reset_index().rename(columns={'index': 'ticker'}),
    on='ticker', how='left'
)

print("\nReorder columns so metadata appears right after ticker")
meta_cols = ['company_name', 'sector', 'founded', 'market_cap_B']
other_cols = [c for c in signals_df.columns if c not in ['ticker'] + meta_cols]
signals_df = signals_df[['ticker'] + meta_cols + other_cols]

print("\nSave signals table to file")
print_report(signals_df, top_pairs, inverse_pairs)
save_signals_csv(signals_df, run_dir / 'signals.csv')

print("\nSave prices for the dashboard interactive charts")
prices_df.to_csv(run_dir / 'prices.csv')

print("\nFetch and save the Fundamental analysis table")
# Fundamental analysis table
fund_raw = fetch_fundamentals(list(prices_df.columns))
fund_df  = score_fundamentals(fund_raw)
save_fundamentals_csv(fund_df, run_dir / 'fundamentals.csv')



Create local folders
  Run outputs → /home/patito/Documents/Inversiones/Bot/V3/outputs/2026-05-08_17-49

Fetching S&P 500 tickers and market_caps
  ✓ 503 tickers fetched from Wikipedia
  Sorting 502 tickers by market cap via yfinance (this takes ~30s)...

Fetching market caps for 502 tickers via yfinance...
  ✓ Market caps retrieved: 502/502
  → Using all 502 S&P 500 tickers

Fetch stock prices

  Tickers with data: 502/502
  ✓ A: 800 bars — last close $115.79
  ✓ AAPL: 800 bars — last close $292.68
  ✓ ABBV: 800 bars — last close $200.67
  ✓ ABNB: 800 bars — last close $145.25
  ✓ ABT: 800 bars — last close $85.38
  ✓ ACGL: 800 bars — last close $94.19
  ✓ ACN: 800 bars — last close $179.08
  ✓ ADBE: 800 bars — last close $251.00
  ✓ ADI: 800 bars — last close $415.36
  ✓ ADM: 800 bars — last close $77.51
  ✓ ADP: 800 bars — last close $212.33
  ✓ ADSK: 800 bars — last close $244.33
  ✓ AEE: 800 bars — last close $109.18
  ✓ AEP: 800 bars — last close $131.57
  ✓ AES: 800 bars — last

In [5]:

print("\nSlice to top N by market cap for a legible heatmap")
top_t = [t for t in tickers if t in corr_matrix.columns][:TOP_N_HIGHLIGHT]
plot_correlation_matrix(corr_matrix.loc[top_t, top_t],
                        save_path=corr_dir / 'correlation_matrix.png')

print("\nGeneral/ — price series highlighted by market cap")
plot_price_series(prices_df, tickers, top_n=TOP_N_HIGHLIGHT, label='market cap',
                  save_path=gen_dir / 'price_series_market-cap.png')

print("\nGeneral/ — price series highlighted by highest absolute stock price")
tickers_by_price = sorted(
    prices_df.columns.tolist(),
    key=lambda t: prices_df[t].iloc[-1],
    reverse=True
)
plot_price_series(prices_df, tickers_by_price, top_n=TOP_N_HIGHLIGHT, label='stock price',
                  save_path=gen_dir / 'price_series_stock-price-absolute.png')

print("\nGeneral/ — price series highlighted by highest normalized return (best performers)")
tickers_by_norm = sorted(
    prices_df.columns.tolist(),
    key=lambda t: prices_df[t].iloc[-1] / prices_df[t].iloc[0],
    reverse=True
)
plot_price_series(prices_df, tickers_by_norm, top_n=TOP_N_HIGHLIGHT, label='normalized return',
                  save_path=gen_dir / 'price_series_normalized-return.png')

print("\nGeneral/ — bar chart: top 15 vs bottom 15 by market cap")
plot_market_cap_bars(prices_df, tickers, market_caps=market_caps, top_n=TOP_N_HIGHLIGHT,
                     save_path=gen_dir / 'market_cap_bars.png')

print("\nGeneral/ — market cap time series (absolute + normalized growth)")
plot_market_cap_series(prices_df, market_caps, top_n=TOP_N_HIGHLIGHT,
                       save_path_abs=gen_dir  / 'market_cap_series_absolute.png',
                       save_path_norm=gen_dir / 'market_cap_series_normalized.png')

print("\nGenerate tables for: top N by predicted return  +  all BUY/SELL tickers")
# Correlation_method/ — per-ticker prediction analysis
# Generate tables for: top N by predicted return  +  all BUY/SELL tickers
top_return_tickers = set(signals_df.head(TOP_N_HIGHLIGHT)['ticker'])
buysell_tickers    = set(signals_df[signals_df['signal'].isin(['BUY', 'SELL'])]['ticker'])
analysis_tickers   = top_return_tickers | buysell_tickers

print("\nGenerating predition analysis plots")
if analysis_tickers:
    print(f"\nGenerating analysis charts ({len(analysis_tickers)} tickers)...")
for ticker in sorted(analysis_tickers):
    pred_ret, r2, top5, corr_signs, y_actual, y_pred = predict_price(
        ticker, returns, corr_matrix
    )
    if y_actual is not None:
        plot_prediction_analysis(
            ticker, returns, prices_df, top5, corr_signs,
            y_actual, y_pred,
            save_path=corr_dir / f'analysis_{ticker}.png'
        )



Slice to top N by market cap for a legible heatmap
  Correlation matrix saved to: /home/patito/Documents/Inversiones/Bot/V3/outputs/2026-05-08_17-49/Correlation_method/correlation_matrix.png

General/ — price series highlighted by market cap
  Price series saved to: /home/patito/Documents/Inversiones/Bot/V3/outputs/2026-05-08_17-49/General/price_series_market-cap.png

General/ — price series highlighted by highest absolute stock price
  Price series saved to: /home/patito/Documents/Inversiones/Bot/V3/outputs/2026-05-08_17-49/General/price_series_stock-price-absolute.png

General/ — price series highlighted by highest normalized return (best performers)
  Price series saved to: /home/patito/Documents/Inversiones/Bot/V3/outputs/2026-05-08_17-49/General/price_series_normalized-return.png

General/ — bar chart: top 15 vs bottom 15 by market cap
  Market cap bars saved to: /home/patito/Documents/Inversiones/Bot/V3/outputs/2026-05-08_17-49/General/market_cap_bars.png

Generate tables for: t

In [6]:
execute_trades = 0
if execute_trades:
    ib = connect_ib()
    try:
        portfolio_value = get_portfolio_value(ib)
        print(f"\nPlacing orders (portfolio: ${portfolio_value:,.0f})...")
        actionable = signals_df[signals_df['signal'].isin(['BUY', 'SELL'])]
        for _, row in actionable.iterrows():
            if row['model_r2'] < MIN_R2:
                continue
            strength = min(1.0, row['model_r2'])
            qty = calculate_position_size(portfolio_value, row['current_price'], strength)
            execute_order(ib, row['ticker'], row['signal'], qty)
    finally:
        ib.disconnect()
        print("\n✓ Disconnected from Interactive Brokers.")
else:
    print("\n  ℹ Simulation mode — no orders placed.")
    print("    To execute on paper trading: run_bot(execute_trades=True)")


  ℹ Simulation mode — no orders placed.
    To execute on paper trading: run_bot(execute_trades=True)
